In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from ocr_vs_vlm.results_final.shared.colors import get_model_color, APPROACH_COLORS
from ocr_vs_vlm.results_final.shared.stats_utils import (
    bootstrap_ci, wilcoxon_test, cohens_d, effect_size_interpretation
)
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template
from ocr_vs_vlm.results_final.shared.data_loader import (
    load_dataset_data, extract_models_from_columns
)

setup_plotly_template()

DATASET = 'publaynet'
print(f"✓ Setup complete for {DATASET}")

✓ Setup complete for publaynet


## 1. Load Data

In [2]:
try:
    df = load_dataset_data(DATASET, task_type='parsing')
    print(f"✓ Loaded {len(df)} samples")
    print(f"  Approaches: {df['approach'].unique().tolist()}")
except Exception as e:
    print(f"Error loading data: {e}")
    df = pd.DataFrame()

models = extract_models_from_columns(df) if len(df) > 0 else []
print(f"  Models: {models}")

Error loading data: load_dataset_data() got an unexpected keyword argument 'task_type'
  Models: []


## 2. Performance Overview

In [3]:
# Compute CER stats
stats = []
for approach in df['approach'].unique() if len(df) > 0 else []:
    app_df = df[df['approach'] == approach]
    for model in models:
        col = f'cer_{model}'
        if col in app_df.columns:
            scores = app_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                stats.append({
                    'Approach': approach,
                    'Model': model,
                    'CER': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi,
                    'Std': np.std(scores),
                    'N': len(scores)
                })

stats_df = pd.DataFrame(stats)
if len(stats_df) > 0:
    display(Markdown("### CER Summary (Lower is Better)"))
    pivot = stats_df.pivot_table(index='Model', columns='Approach', values='CER').round(4)
    display(pivot)

In [4]:
# Multi-metric view if available (CER and WER)
if len(df) > 0:
    metrics = ['cer', 'wer']
    available_metrics = []
    
    for metric in metrics:
        if any(metric in c for c in df.columns):
            available_metrics.append(metric)
    
    if len(available_metrics) > 1:
        fig = make_subplots(rows=1, cols=len(available_metrics),
                           subplot_titles=[m.upper() for m in available_metrics])
        
        for i, metric in enumerate(available_metrics):
            for model in models:
                col = f'{metric}_{model}'
                if col in df.columns:
                    scores = df[col].dropna()
                    if len(scores) > 0:
                        fig.add_trace(
                            go.Box(y=scores, name=model,
                                  marker_color=get_model_color(model),
                                  showlegend=(i == 0)),
                            row=1, col=i+1
                        )
        
        fig.update_layout(
            title=f'{DATASET}: Multi-Metric Comparison',
            height=450,
            boxmode='group'
        )
        fig.show()
    else:
        # Single metric view
        if len(stats_df) > 0:
            fig = go.Figure()
            
            for model in models:
                model_df = stats_df[stats_df['Model'] == model]
                if len(model_df) > 0:
                    fig.add_trace(go.Bar(
                        x=model_df['Approach'],
                        y=model_df['CER'],
                        name=model,
                        marker_color=get_model_color(model)
                    ))
            
            fig.update_layout(
                title=f'{DATASET}: CER by Model and Approach',
                yaxis_title='CER',
                barmode='group',
                height=450
            )
            fig.show()

## 3. Statistical Comparison

In [5]:
# Pairwise approach comparison
if len(df) > 0:
    approach_scores = {}
    
    for approach in df['approach'].unique():
        app_df = df[df['approach'] == approach]
        all_scores = []
        for model in models:
            col = f'cer_{model}'
            if col in app_df.columns:
                all_scores.extend(app_df[col].dropna().tolist())
        approach_scores[approach] = np.array(all_scores)
    
    print("📊 Statistical Comparison")
    print("=" * 60)
    
    approaches = list(approach_scores.keys())
    for i, a1 in enumerate(approaches):
        for a2 in approaches[i+1:]:
            if len(approach_scores[a1]) > 0 and len(approach_scores[a2]) > 0:
                n = min(len(approach_scores[a1]), len(approach_scores[a2]))
                stat, p = wilcoxon_test(approach_scores[a1][:n], approach_scores[a2][:n])
                d = cohens_d(approach_scores[a1], approach_scores[a2])
                
                m1, m2 = np.mean(approach_scores[a1]), np.mean(approach_scores[a2])
                winner = a1 if m1 < m2 else a2
                
                print(f"\n{a1} vs {a2}:")
                print(f"   Mean CER: {m1:.4f} vs {m2:.4f}")
                print(f"   Winner: {winner}")
                print(f"   p-value: {p:.4f} | d={d:.3f} ({effect_size_interpretation(d)})")

## 4. Document Type Analysis

In [6]:
# If document categories exist
if len(df) > 0 and 'category' in df.columns:
    display(Markdown("### Performance by Document Category"))
    
    category_stats = []
    for category in df['category'].unique():
        cat_df = df[df['category'] == category]
        for model in models:
            col = f'cer_{model}'
            if col in cat_df.columns:
                scores = cat_df[col].dropna()
                if len(scores) > 0:
                    category_stats.append({
                        'Category': category,
                        'Model': model,
                        'Mean CER': scores.mean(),
                        'N': len(scores)
                    })
    
    cat_df = pd.DataFrame(category_stats)
    if len(cat_df) > 0:
        display(cat_df.pivot_table(index='Category', columns='Model', values='Mean CER').round(4))
else:
    print("No category information available")

No category information available


## 5. Key Findings

In [7]:
print("=" * 60)
print(f"{DATASET.upper()} KEY FINDINGS")
print("=" * 60)

if len(stats_df) > 0:
    best = stats_df.loc[stats_df['CER'].idxmin()]
    print(f"\n🏆 Best Configuration:")
    print(f"   {best['Approach']} + {best['Model']}: CER = {best['CER']:.4f}")
    print(f"   95% CI: [{best['CI_Lower']:.4f}, {best['CI_Upper']:.4f}]")
    
    approach_means = stats_df.groupby('Approach')['CER'].mean().sort_values()
    print(f"\n📊 Approach Ranking (Lower CER is Better):")
    for i, (app, score) in enumerate(approach_means.items(), 1):
        print(f"   {i}. {app}: {score:.4f}")
    
    print("\n💡 PubLayNet-Specific Observations:")
    print("   - Scientific document layout is relatively structured")
    print("   - Tables and figures may present challenges")
    print("   - Multi-column layouts affect extraction accuracy")

print("\n" + "=" * 60)

PUBLAYNET KEY FINDINGS

